In [1]:
import warnings
warnings.filterwarnings("ignore")
import os, gc, math, pathlib
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM  #type: ignore
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [1]:
import torch, psutil, subprocess

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

print("CPU cores:", psutil.cpu_count())
print("RAM (GB):", psutil.virtual_memory().total / 1e9)

print("\nDisk space:")
# subprocess.run(["df", "-h", "/teamspace/studios/this_studio"])

/teamspace/studios/this_studio/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


GPU available: True
GPU: Tesla T4
VRAM (GB): 15.636037632
CPU cores: 8
RAM (GB): 31.538065408

Disk space:


In [7]:
# pip install "numpy<2" "transformers>=4.52.1" llmcompressor compressed-tensors datasets torch --upgrade


In [3]:
import torch
import transformers
import llmcompressor
from llmcompressor.modifiers.quantization import GPTQModifier

print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")

# Verify recipe initialization
recipe = GPTQModifier(
    scheme="W4A16",
    targets="Linear",
    ignore=["lm_head"],
)

print(f"\nRecipe successfully loaded:\n{recipe}")

PyTorch version: 2.5.1+cu121
Transformers version: 4.52.4

Recipe successfully loaded:
config_groups=None targets=['Linear'] ignore=['lm_head'] scheme='W4A16' kv_cache_scheme=None index=None group=None start=None end=None update=None initialized_=False finalized_=False started_=False ended_=False sequential_update=True sequential_targets=None block_size=128 dampening_frac=0.01 actorder=None offload_hessians=False


In [5]:
import os
from huggingface_hub import snapshot_download

MODEL_DIR = "/teamspace/studios/this_studio/models/Qwen3-0.6B"
OUTPUT_DIR = "/teamspace/studios/this_studio/models/Qwen3-0.6B-W4A16"

os.makedirs(MODEL_DIR, exist_ok=True)

snapshot_download(
    repo_id="Qwen/Qwen3-0.6B",
    local_dir=MODEL_DIR,
)
print(f"Model downloaded to: {MODEL_DIR}")

Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 376.17it/s]

Model downloaded to: /teamspace/studios/this_studio/models/Qwen3-0.6B


In [ ]:
from llmcompressor import oneshot

if not os.path.isdir(OUTPUT_DIR):
    oneshot(
        model="Qwen/Qwen3-0.6B",
        dataset="wikitext",
        dataset_config_name="wikitext-2-raw-v1",
        recipe=recipe,
        output_dir=OUTPUT_DIR,
        max_seq_length=4096,
        num_calibration_samples=256,
    )
    print(f"Quantization complete. Model saved to: {OUTPUT_DIR}")

Tokenizing: 100%|██████████| 3760/3760 [00:05<00:00, 679.04 examples/s]

2026-07-30T02:48:20.624006+0000 | reset | INFO - Compression lifecycle reset
2026-07-30T02:48:20.628999+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-07-30T02:48:20.768820+0000 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-07-30T02:48:20.770192+0000 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`
2026-07-30T02:48:20.774629+0000 | dispatch_for_sequential | WARNING - CUDA is not available! Compressing model on CPU instead



(1/29): Calibrating:  51%|█████     | 131/256 [1:42:29<1:33:55, 45.09s/it]

In [6]:
import torch
from transformers import AutoModelForCausalLM
from llmcompressor import oneshot

# 1. Load model directly onto GPU
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B",
    torch_dtype="auto",
    device_map="cuda", # <--- Forces CUDA GPU execution
)

# 2. Run oneshot with optimized calibration sequence length
oneshot(
    model=model,
    dataset="wikitext",
    dataset_config_name="wikitext-2-raw-v1",
    recipe=recipe,
    output_dir=OUTPUT_DIR,
    max_seq_length=1024,          # <--- Reduced from 4096 to 1024 (4x speedup)
    num_calibration_samples=256,
)

print(f"Quantization complete! Model saved to: {OUTPUT_DIR}")

Tokenizing: 100%|██████████| 3760/3760 [00:03<00:00, 1132.35 examples/s]

2026-07-30T11:22:11.593097+0000 | reset | INFO - Compression lifecycle reset
2026-07-30T11:22:11.598309+0000 | from_modifiers | INFO - Creating recipe from modifiers


2026-07-30T11:22:11.704411+0000 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-07-30T11:22:11.706044+0000 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`


(1/29): Calibrating: 100%|██████████| 256/256 [00:08<00:00, 31.41it/s]

2026-07-30T11:22:22.250101+0000 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 256 samples


...

2026-07-30T11:31:38.081450+0000 | compress | METRIC - time 0.74s
2026-07-30T11:31:38.082647+0000 | compress | METRIC - error 373690.12
2026-07-30T11:31:38.084215+0000 | compress | METRIC - GPU 0 | usage: 13.39% | total memory: 16 GB
2026-07-30T11:31:38.085650+0000 | compress | METRIC - Compressed module size: 4.243456 MB
2026-07-30T11:31:38.087083+0000 | compress_modules | INFO - Quantizing model.layers.27.self_attn.k_proj using 256 samples
2026-07-30T11:31:38.692195+0000 | compress | METRIC - time 0.60s
2026-07-30T11:31:38.693391+0000 | compress | METRIC - error 179236.97
2026-07-30T11:31:38.694853+0000 | compress | METRIC - GPU 0 | usage: 13.39% | total memory: 16 GB
2026-07-30T11:31:38.695559+0000 | compress | METRIC - Compressed module size: 2.121728 MB
2026-07-30T11:31:38.696920+0000 | compress_modules | INFO - Quantizing model.layers.27.self_attn.v_proj using 256 samples
2026-07-30T11:31:39.429237+0000 | compress | METRIC - time 0.73s
2026-07-30T11:31:39.431078+0000 | compress | 

(29/29): Propagating: 100%|██████████| 256/256 [00:36<00:00,  7.04it/s]


2026-07-30T11:33:01.923144+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-07-30T11:33:02.494109+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 427it [00:04, 100.85it/s]


Quantization complete! Model saved to: /teamspace/studios/this_studio/models/Qwen3-0.6B-W4A16
